###  Query Similarity Analysis
Find similar SPARQL queries by comparing their DBpedia result sets (Jaccard) and query texts (Cosine)

In [1]:
import sys
import os
from tqdm.auto import tqdm

sys.path.append('.')
from utils.query_loader import QueryLoader
from utils.jaccard_similarity import JaccardSimilarity
from utils.cosine_similarity import CosineSimilarity
from utils.results_analyzer import ResultsAnalyzer
from utils.performance_monitor import PerformanceMonitor
from utils.config import Config


/upb/users/z/zahera/profiles/unix/cs/.conda/envs/llm-gpu-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Configuration:

In [2]:
import torch

Config.INPUT_FILE = os.path.join("queries", "successful_queries.json")
Config.setup_output_dir()
Config.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {Config.DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

monitor = PerformanceMonitor()
monitor.checkpoint("Initialization")


Device: cuda
GPU: NVIDIA A100 80GB PCIe

📊 CHECKPOINT: Initialization
   Time: 22:51:33
   Memory: 0.68 GB
   CPU: 0.0%
   Elapsed: 0.5 seconds
   GPU Memory: 0.00 GB


{'stage': 'Initialization',
 'memory_gb': 0.6793670654296875,
 'cpu_percent': 0.0,
 'elapsed_seconds': 0.5008664131164551,
 'timestamp': '22:51:33'}

###  Load Queries:

In [3]:
import time

loader = QueryLoader(max_entries=Config.MAX_QUERIES, show_progress=True)

print(f"Loading up to {Config.MAX_QUERIES:,} queries...")
start_time = time.time()

preprocessed = loader.load_smart(Config.INPUT_FILE)

elapsed = time.time() - start_time
print(f"✓ Loaded {len(preprocessed):,} queries in {elapsed/60:.1f} min")
print(f"  Sample: {preprocessed[0][1][:100]}...")

monitor.checkpoint("Query Loading")


Loading up to 150,000 queries...
🤔 Analyzing queries/successful_queries.json...
📁 Large file detected (85.9 GB), using streaming...
📥 Streaming queries from: queries/successful_queries.json
📊 Max entries: 150,000
📁 File size: 87995.6 MB


📥 Streaming queries: 100%|██████████| 150000/150000 [19:11<00:00, 130.26queries/s, Processed=150,000] 


✅ Successfully loaded 150,000 queries
⏱️  Loading time: 1151.5 seconds (19.2 minutes)
⚡ Speed: 130.3 queries/second


✓ Loaded 150,000 queries in 19.2 min
  Sample: PREFIX  dbpo: <http://dbpedia.org/ontology/>
PREFIX  rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-...

📊 CHECKPOINT: Query Loading
   Time: 23:10:49
   Memory: 11.23 GB
   CPU: 0.0%
   Elapsed: 1156.5 seconds
   GPU Memory: 0.00 GB


{'stage': 'Query Loading',
 'memory_gb': 11.229045867919922,
 'cpu_percent': 0.0,
 'elapsed_seconds': 1156.491305589676,
 'timestamp': '23:10:49'}

### Compute Jaccard Similarities: 

In [4]:
jaccard_calculator = JaccardSimilarity(
    threshold=Config.JACCARD_THRESHOLD,
    max_similars=Config.MAX_SIMILARS_PER_QUERY,
    num_perm=Config.MINHASH_PERMUTATIONS
)

print(f"Computing Jaccard similarities (threshold={Config.JACCARD_THRESHOLD})...")
jaccard_results = jaccard_calculator.compute_similarities(preprocessed)

output_files = Config.get_output_files(len(preprocessed))
jaccard_calculator.save_results(jaccard_results, output_files["jaccard"])

print(f"✓ Found {len(jaccard_results):,} queries with similar results")
monitor.checkpoint("Jaccard Computation")


🎯 JaccardSimilarity initialized:
   Threshold: 0.9
   Max similars per query: 20
   MinHash permutations: 64
Computing Jaccard similarities (threshold=0.9)...

🔍 Step 2: Finding similar pairs (threshold=0.9)...

🔨 Step 1: Building LSH index...
   Creating MinHash signatures...


   Hashing results: 100%|██████████| 150000/150000 [05:14<00:00, 476.81queries/s] 

   ✓ Created 150,000 MinHash signatures
   Building LSH index...



   Inserting into LSH: 100%|██████████| 150000/150000 [00:03<00:00, 40101.03signatures/s]

✅ LSH index built with 150,000 signatures in 318.3s



   Querying for similarities:  91%|█████████ | 136761/150000 [00:47<00:12, 1057.44queries/s, found=1e+4, progress=137018/150000]

✅ Results saved in 1.3s
📁 File size: 102.6 MB
📄 Location: results/similarity_analysis_150000_jaccard_thresh_0.9.json
✓ Found 11,791 queries with similar results

📊 CHECKPOINT: Jaccard Computation
   Time: 23:17:21
   Memory: 11.84 GB
   CPU: 2.0%
   Elapsed: 1548.0 seconds
   GPU Memory: 0.00 GB


{'stage': 'Jaccard Computation',
 'memory_gb': 11.841300964355469,
 'cpu_percent': 2.0,
 'elapsed_seconds': 1548.0390710830688,
 'timestamp': '23:17:21'}

### Compute Jaccard Similarities:

In [5]:
if not jaccard_results:
    print("⚠ No Jaccard similarities found")
    cosine_results = []
else:
    cosine_calculator = CosineSimilarity(
        model_name=Config.EMBEDDING_MODEL,
        device=Config.DEVICE
    )
    
    print(f"Computing cosine similarities...")
    cosine_results = cosine_calculator.compute_similarities(
        jaccard_results,
        batch_size=Config.BATCH_SIZE_EMBEDDINGS
    )
    
    cosine_calculator.save_results(cosine_results, output_files["cosine"])
    cosine_calculator.cleanup()
    
    print(f"✓ Processed {len(cosine_results):,} entries")
    monitor.checkpoint("Cosine Computation")


Computing cosine similarities...
Loading embedding model: all-MiniLM-L6-v2
Using device: cuda
Computing cosine similarities for 11791 entries...


Processing cosine batches: 100%|██████████| 93/93 [01:17<00:00,  1.20it/s, Processed=11776, Remaining=15]  



✓ Cosine computation complete. Processed 11791 entries
✓ Cosine results saved to: results/similarity_analysis_150000_cosine.json
✓ Processed 11,791 entries

📊 CHECKPOINT: Cosine Computation
   Time: 23:20:17
   Memory: 12.37 GB
   CPU: 0.0%
   Elapsed: 1724.9 seconds
   GPU Memory: 0.01 GB


###  Analysis and Export:

In [ ]:
# Cell 6: Analysis and Export
analyzer = ResultsAnalyzer()
analyzer.analyze_similarities(output_files["cosine"])
analyzer.print_summary()

df = analyzer.export_to_csv(output_files["cosine"], output_files["csv"])
analyzer.save_summary(output_files["summary"])

monitor.checkpoint("Analysis & Export")


📊 Analyzing results from results/similarity_analysis_150000_cosine.json...


   Analyzing entries: 100%|██████████| 11791/11791 [00:03<00:00, 3561.75entries/s, With similars=11791, Pairs=97314]




📊 RESULTS SUMMARY

📈 Basic Statistics:
   Total base queries: 11,791
   Queries with similar results: 11,791
   Percentage with similars: 100.0%
   Total similar pairs found: 97,314

🎯 Jaccard Similarity:
   Mean: 0.9903
   Std Dev: 0.0252
   Min: 0.9000
   Max: 1.0000

📐 Cosine Similarity:
   Mean: 0.9198
   Std Dev: 0.0542
   Min: 0.5199
   Max: 1.0000

⚖️  Combined Score (Jaccard + Cosine / 2):
   Mean: 0.9550
   Std Dev: 0.0304
   Min: 0.7599
   Max: 1.0000

📋 Distribution by Jaccard Range:
   jaccard_0.9: 13,729 pairs (14.1%)
   jaccard_1.0: 83,585 pairs (85.9%)
📁 Exporting to CSV: results/similarity_analysis_150000_results.csv


   Preparing CSV data:  25%|██▍       | 2938/11791 [00:00<00:02, 4012.49entries/s, Rows=28973]

### Example Similar Pairs: 

In [8]:
print("\n🔍 Example Similar Pairs:")
examples = analyzer.get_examples(output_files["cosine"], n_examples=3, n_similars=2)

for i, ex in enumerate(examples, 1):
    print(f"\nExample {i}: {ex['base_query'][:80]}...")
    for j, sim in enumerate(ex['similar_pairs'], 1):
        print(f"  Jaccard={sim['jaccard_score']:.3f}, "
              f"Cosine={sim.get('cosine_score', 0):.3f}")



🔍 Example Similar Pairs:
🔍 Getting 3 example pairs...
✅ Found 3 examples

Example 1: PREFIX  rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX  owl:  <http:...
  Jaccard=1.000, Cosine=0.975
  Jaccard=1.000, Cosine=0.971

Example 2: PREFIX  rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX  dbpo: <http:...
  Jaccard=1.000, Cosine=0.966
  Jaccard=1.000, Cosine=0.964

Example 3: PREFIX  rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX  owl:  <http:...
  Jaccard=1.000, Cosine=0.981
  Jaccard=1.000, Cosine=0.978


### Cleanup:

In [9]:
import gc

# Cleanup memory
if 'jaccard_calculator' in locals():
    jaccard_calculator.cleanup()
if 'cosine_calculator' in locals():
    cosine_calculator.cleanup()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Final summary
monitor.print_summary()
print(f"\n✓ Analysis complete!")
print(f"  Processed: {len(preprocessed):,} queries")
print(f"  Similar pairs: {analyzer.stats.get('total_similar_pairs', 0):,}")
print(f"  Output: {Config.OUTPUT_DIR}")


🧹 Memory cleaned up

PERFORMANCE SUMMARY

⏱️  Total time: 2614.2 seconds (43.6 minutes)

📈 Peak Usage:
   Peak Memory: 12.48 GB
   Peak CPU: 2.0%

📋 Stage-by-stage:
   Initialization: 0.5s, 0.68GB, 0.0% CPU
   Query Loading: 1156.5s, 11.23GB, 0.0% CPU
   Jaccard Computation: 1548.0s, 11.84GB, 2.0% CPU
   Cosine Computation: 1724.9s, 12.37GB, 0.0% CPU
   Analysis & Export: 1949.8s, 12.48GB, 2.0% CPU

✓ Analysis complete!
  Processed: 150,000 queries
  Similar pairs: 97,314
  Output: results
